In [0]:
import pandas as pd                                                     # Import pandas for data cleaning
import seaborn as sns                                                   # Import Seaborn for visualization
import matplotlib.pyplot as plt                                         # Import matplotlib library for visualization
import plotly.express as px                                             # Import plotly library for visualization
import plotly.graph_objects as go

from pyspark.sql import functions as F
from pyspark.sql.functions import col                                   # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev,count          # MathsFunctions
from pyspark.sql.functions import to_date,month,year,datediff           # DateFunctions
from pyspark.sql.functions import abs                                   # OtherFunctions


In [0]:
orders = spark.read.table('samplesuperstore.bronzedata.orders')
people = spark.read.table('samplesuperstore.bronzedata.people')
returns = spark.read.table('samplesuperstore.bronzedata.returns')
 
# #Ingetion data from databricks catelogs-samplesuperstore.bronzedata

In [0]:
# Build summary for each column
columns = orders.columns
summary_data = []

# Get total record count once
total_records = orders.count()

for column in columns:
    not_nulls = orders.filter(F.col(column).isNotNull()).count()
    nulls = orders.filter(F.col(column).isNull()).count()
    summary_data.append((column, total_records, not_nulls, nulls))

summary_df = spark.createDataFrame(
    summary_data,
    ["Column", "AllValues", "NotNulls", "Nulls"]
)

# Filter columns with nulls > 0
summary_df = summary_df.filter(col("Nulls") > 0)

# Display summary ordered by Nulls descending
display(summary_df.orderBy("Nulls", ascending=False))

# Replace nulls with 0 in Spark DataFrame
df = orders.na.fill(0)

# Print total number of columns with nulls
print("Columns with Nulls:", summary_df.count())

Column,AllValues,NotNulls,Nulls


Columns with Nulls: 0


In [0]:
total_records = orders.count()

summary_df = spark.createDataFrame(
    [
        (
            "Quantity",
            total_records,
            orders.filter(F.col("Quantity").isNotNull()).count(),
            orders.filter(F.col("Quantity").isNull()).count(),
            orders.approxQuantile("Quantity", [0.5], 0.01)[0],
            orders.agg(F.avg("Quantity")).first()[0],
            orders.agg(F.min("Quantity")).first()[0],
            orders.agg(F.max("Quantity")).first()[0],
            orders.agg(F.stddev("Quantity")).first()[0],
            orders.agg(F.sum("Quantity")).first()[0]          
        ),
        (
            "Sales",
            total_records,
            orders.filter(F.col("Sales").isNotNull()).count(),
            orders.filter(F.col("Sales").isNull()).count(),
            orders.approxQuantile("Sales", [0.5], 0.01)[0],
            orders.agg(F.avg("Sales")).first()[0],
            orders.agg(F.min("Sales")).first()[0],
            orders.agg(F.max("Sales")).first()[0],
            orders.agg(F.stddev("Sales")).first()[0],
            orders.agg(F.sum("Sales")).first()[0]
        ),

        (
            "Profit",
            total_records,
            orders.filter(F.col("Profit").isNotNull()).count(),
            orders.filter(F.col("Profit").isNull()).count(),
            orders.approxQuantile("Profit", [0.5], 0.01)[0],
            orders.agg(F.avg("Profit")).first()[0],
            orders.agg(F.min("Profit")).first()[0],
            orders.agg(F.max("Profit")).first()[0],
            orders.agg(F.stddev("Profit")).first()[0],
            orders.agg(F.sum("Profit")).first()[0]
        )
    ],
    ["Column", "AllCount", "NoNulls", "Nulls","Median","Mean","Min", "Max", "StdDev","Sum"]
)

#display(summary_df)


In [0]:

# Create temp table for Outlier Stats
Outliers_Orders = orders.select("Order ID", "Sales")

# Calculate quartiles and fences using approxQuantile
Q1 = Outliers_Orders.approxQuantile("Sales", [0.25], 0.01)[0]
Q3 = Outliers_Orders.approxQuantile("Sales", [0.75], 0.01)[0]
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Add SalesOutlier column (1 = Yes, 0 = No) → keep all rows
Outliers_Orders = Outliers_Orders.withColumn(
    "SalesOutlier",
    F.when((F.col("Sales") < lower_fence) | (F.col("Sales") > upper_fence), F.lit(1)).otherwise(F.lit(0))
)

# Compute mean and stddev
stats = Outliers_Orders.select(
    F.mean("Sales").alias("mean"),
    F.stddev("Sales").alias("stddev")
).collect()[0]

mean_sales = stats["mean"]
std_sales = stats["stddev"]

# Add Z-score and anomaly flag
Outliers_Orders = Outliers_Orders.withColumn(
    "Sales_Zscore", (F.col("Sales") - mean_sales) / std_sales
).withColumn(
    "SalesAnomaly", F.when(F.abs(F.col("Sales_Zscore")) > 3, 1).otherwise(0)
)

display(Outliers_Orders)


# Save as table in silverdata
# Outliers_Orders.write.mode("overwrite").saveAsTable(samplesuperstore.silverdata.EDA_Orders_Outliers)

# Histogram
fig = px.histogram(Outliers_Orders, x="Sales", nbins=50, title="Sales Distribution with Outlier Fences")

# Add vertical lines for fences
fig.add_vline(x=lower_fence, line_dash="dash", line_color="red", annotation_text="Lower Fence")
fig.add_vline(x=upper_fence, line_dash="dash", line_color="green", annotation_text="Upper Fence")

# fig.show()


#display(Outliers_Orders.orderBy(F.col("Sales").desc()))


Order ID,Sales,SalesOutlier,Sales_Zscore,SalesAnomaly
CA-2017-126536,0.99,0,-0.3672198957419812,0
CA-2015-102015,1.24,0,-0.36681876944384273,0
CA-2016-111682,1.68,0,-0.366112787159119,0
CA-2016-163153,1.344,0,-0.36665190090381716,0
CA-2014-154158,1.344,0,-0.36665190090381716,0
CA-2014-143168,1.344,0,-0.36665190090381716,0
CA-2017-117702,1.64,0,-0.3661769673668212,0
CA-2017-129378,1.44,0,-0.36649786840533194,0
US-2017-127292,2.04,0,-0.36553516528979957,0
US-2014-168501,1.632,0,-0.3661898034083616,0
